#  End-to-End PySpark Data Engineering Project

In [29]:
# Installing Java (Spark requires Java to run)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [30]:
!pip install -q pyspark findspark

In [31]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
# Creation of Spark session
spark = SparkSession.builder \
    .appName("OpenML_Project") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


##Task 1: Data Ingestion & Exploration

In [32]:
# Load Dataset
df = spark.read.csv("/content/BNPParibas_Data.csv", header=True, inferSchema=True)

In [33]:
df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|          2| 69|           64|           24.2|       1613.6|Month-to-Month|           Fiber|              1|   Credit Card|    1|
|          3| 46|           28|          78.02|      2053.63|      One Year|           Fiber|              1|           UPI|    0|
|          4| 32|           39|          45.95|      1688.47|Month-to-Month|           Fiber|              1|           UPI|    0|
|          5| 60|           57|          49.88|      2950.23|Month-to-Month|       

In [34]:
# Display schema
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



In [35]:
# Display record count
record_count = df.count()
print(f"Total Record Count: {record_count}")

Total Record Count: 1000


In [49]:
# Display Null count
from pyspark.sql.functions import *
print("Null counts:")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()


Null counts:
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



In [37]:
# Display Duplicate count
duplicates = df.count() - df.distinct().count()
print(f"Duplicate Count: {duplicates}")

Duplicate Count: 0


In [38]:
# Display Datatypes
print("Datatypes: ")
df.dtypes

Datatypes: 


[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

In [39]:
# Summary statistics
print("Summary Statistics:")
df.describe().show()

Summary Statistics:
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|summary|      customer_id|               age|     tenure_months|  monthly_charges|     total_charges| contract_type|internet_service|   support_tickets|payment_method|             churn|
+-------+-----------------+------------------+------------------+-----------------+------------------+--------------+----------------+------------------+--------------+------------------+
|  count|             1000|              1000|              1000|             1000|              1000|          1000|            1000|              1000|          1000|              1000|
|   mean|            500.5|            43.819|            35.459|79.96715000000002| 2800.235379999997|          NULL|            NULL|             1.956|          NULL|             0.502|
| stddev|288.8194360957494|14.9910296500

## Task 2: ETL Pipeline Development

#### Extract : Read source dataset.

In [40]:
import os

raw_df = spark.read.csv("/content/BNPParibas_Data.csv", header=True, inferSchema=True)

#### Transform
Perform:
- Missing Value Treatment
- Duplicate Removal
- Data Type Conversion
- Feature Engineering
- Aggregation

In [41]:
# Duplicate Removal
transformed_df = raw_df.dropDuplicates()

transformed_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [42]:
# Missing Value Treatment (Imputing numeric columns with mean, filling strings with 'Unknown')
numeric_cols = [f.name for f in transformed_df.schema.fields if str(f.dataType) in ["IntegerType", "DoubleType"]]
string_cols = [f.name for f in transformed_df.schema.fields if str(f.dataType) == "StringType"]

if string_cols:
    transformed_df = transformed_df.fillna("Unknown", subset=string_cols)
for col_name in numeric_cols:
    mean_val = transformed_df.select(mean(col(col_name))).first()[0]
    if mean_val is not None:
        transformed_df = transformed_df.fillna({col_name: mean_val})

transformed_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [43]:
# Data Type Conversion
transformed_df = transformed_df.withColumn("churn", col("churn").cast("int")) \
                               .withColumn("total_charges", col("total_charges").cast("double")) \
                               .withColumn("age", col("age").cast("integer"))

transformed_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|       

In [44]:
# Feature Engineering
# Create a new feature 'is_adult' based on age
transformed_df = transformed_df.withColumn("is_adult", when(col("age") >= 18, True).otherwise(False))
# Create 'charge_per_tenure' to see how much they pay relative to their tenure
transformed_df = transformed_df.withColumn("charge_per_tenure", round(col("total_charges") / (col("tenure_months") + 1),3))

transformed_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|    true|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    true|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|    true|            27.64|
|        410| 19|           51|          88.61|     

In [46]:
# Aggregation
# Calculate average monthly charges by contract type
contract_avg_df = transformed_df.groupBy("contract_type").agg(avg("monthly_charges").alias("avg_contract_monthly_charges"))
contract_avg_df.show()

+--------------+----------------------------+
| contract_type|avg_contract_monthly_charges|
+--------------+----------------------------+
|Month-to-Month|           80.42109777015438|
|      One Year|           80.04780575539571|
|      Two Year|           77.90187050359717|
+--------------+----------------------------+



#### Load
Store transformed data in:
silver_layer/
using Parquet format.

In [47]:
target_directory = "/content/Data/silver_layer"

# Saving the transformed data in optimized Parquet format
transformed_df.write \
    .mode("overwrite") \
    .parquet(target_directory)

print(f"ETL Pipeline executed successfully! Output saved to local path: '{target_directory}/'")

ETL Pipeline executed successfully! Output saved to local path: '/content/Data/silver_layer/'


### ETL flow diagram
- Extract : Extracting the raw data from dataset.
- Transform : Applying transform logic on raw_df or we can say bronze layer to get cleaned dataframe.
- Load : Creating a Data named folder in colab file system and giving target directory where we upload cleaned dataset as named silver_layer.

#### Output Dataset verfication


In [48]:
silver_df = spark.read.parquet("/content/Data/silver_layer")
silver_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|    true|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    true|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|    true|            27.64|
|        410| 19|           51|          88.61|     

## Task 3: ELT Pipeline & Medallion Architecture
#### Objective:
- Implement Bronze-Silver-Gold architecture.

In [50]:
# Creating simple local folders in colab for medallion architectue.
os.makedirs("bronze", exist_ok=True)
os.makedirs("silver", exist_ok=True)
os.makedirs("gold", exist_ok=True)

In [52]:
# raw data
raw_df = spark.read.csv("/content/BNPParibas_Data.csv", header=True, inferSchema=True)

#### Bronze Layer : Raw Data
- bronze/

In [53]:
# Saving exact copy in broze folder for bronze layer of architecture.
raw_df.write.mode("overwrite").parquet("bronze")

#### Silver Layer : Cleaned Data
- silver/

In [55]:
# for silver layer we have already applied various transformation above so read the exact data from silver_layer
silver_df = spark.read.parquet("/content/Data/silver_layer")

# Saving this tranformed and cleaned data in silver folder for silver layer of architecture.
silver_df.write.mode("overwrite").parquet("silver")

#### Gold Layer : Business Ready Dataset
gold/

In [56]:
# Reading from silver layer
df_silver = spark.read.parquet("silver")
df_silver.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|    true|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    true|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|    true|            27.64|
|        410| 19|           51|          88.61|     

##### Generating business KPIs.

In [57]:
# Churn Distribution
kpi1_churn_dist = df_silver.groupBy("churn").count().withColumnRenamed("count", "total_customers")
kpi1_churn_dist.show()

+-----+---------------+
|churn|total_customers|
+-----+---------------+
|    1|            502|
|    0|            498|
+-----+---------------+



In [58]:
# Average Monthly Charges by Churn
kpi2_avg_charges = df_silver.groupBy("churn").agg(avg("monthly_charges").alias("avg_monthly_charges"))
kpi2_avg_charges.show()

+-----+-------------------+
|churn|avg_monthly_charges|
+-----+-------------------+
|    1|  86.39589641434263|
|    0|  73.48676706827304|
+-----+-------------------+



In [59]:
# Churn by Contract Type
kpi3_contract_churn = df_silver.groupBy("contract_type", "churn").count()
kpi3_contract_churn.show()

+--------------+-----+-----+
| contract_type|churn|count|
+--------------+-----+-----+
|      Two Year|    1|   30|
|      One Year|    0|  211|
|Month-to-Month|    1|  405|
|      Two Year|    0|  109|
|Month-to-Month|    0|  178|
|      One Year|    1|   67|
+--------------+-----+-----+



In [60]:
# Saving KPIs to Gold folder for Gold layer of architecture
kpi1_churn_dist.write.mode("overwrite").parquet("gold")
kpi2_avg_charges.write.mode("overwrite").parquet("gold")
kpi3_contract_churn.write.mode("overwrite").parquet("gold")

## Task 4: PySpark + Pandas Integration
#### Objective :
- Demonstrate interoperability between Pandas and Spark.

Convert Spark → Pandas :
- df.toPandas()

In [61]:
import pandas as pd

# Converting spark dataframe to pandas dataframe.
pandas_df = silver_df.toPandas()

#### Feature Engineering in Pandas - Create:
- Ratio Column
- Percentage Column
- Growth Metric

In [64]:
# Ratio column
pandas_df['Average_monthly_charge']=(pandas_df['total_charges']/ (pandas_df['tenure_months']+1)).round(2)
pandas_df.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_adult,charge_per_tenure,Average_monthly_charge,revenue_percentage_contribution
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,True,58.101,58.10,0.033198
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,True,83.251,83.25,0.115946
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,True,27.640,27.64,0.040469
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,True,84.008,84.01,0.156001
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,True,136.339,136.34,0.233704


In [65]:
# Percentage column
total_revenue_sum = pandas_df['total_charges'].sum()
pandas_df['revenue_percentage_contribution'] = (pandas_df['total_charges'] / total_revenue_sum) * 100
pandas_df.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_adult,charge_per_tenure,Average_monthly_charge,revenue_percentage_contribution
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,True,58.101,58.10,0.033198
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,True,83.251,83.25,0.115946
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,True,27.640,27.64,0.040469
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,True,84.008,84.01,0.156001
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,True,136.339,136.34,0.233704


In [66]:
# Growth Metric
avg_monthly_charge = pandas_df['monthly_charges'].mean()
pandas_df['monthly_charge_variance'] = pandas_df['monthly_charges'] - avg_monthly_charge
pandas_df.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,is_adult,charge_per_tenure,Average_monthly_charge,revenue_percentage_contribution,monthly_charge_variance
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,True,58.101,58.10,0.033198,-20.73715
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,True,83.251,83.25,0.115946,0.89285
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,True,27.640,27.64,0.040469,-50.05715
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,True,84.008,84.01,0.156001,8.64285
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,True,136.339,136.34,0.233704,61.39285


Convert Pandas → Spark :
- spark.createDataFrame(pdf)

In [67]:
# Converting pandas back to Spark
spark_df=spark.createDataFrame(pandas_df)
spark_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------------+-------------------------------+-----------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|Average_monthly_charge|revenue_percentage_contribution|monthly_charge_variance|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------------+-------------------------------+-----------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|    true|           58.101|                  58.1|            0.03319792352598588|    -20.737149999999993|
|        182| 23|           38|          80.86|     

In [68]:
spark_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- tenure_months: long (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: long (nullable = true)
 |-- is_adult: boolean (nullable = true)
 |-- charge_per_tenure: double (nullable = true)
 |-- Average_monthly_charge: double (nullable = true)
 |-- revenue_percentage_contribution: double (nullable = true)
 |-- monthly_charge_variance: double (nullable = true)



## Task 5: Spark SQL
#### Objective :
- Generate analytical reports using Spark SQL.

In [69]:
from pyspark.sql.window import Window

# Creating temporary view
df_silver.createOrReplaceTempView('my_view')

### SQL_QUERY

In [70]:
# Query 1 : Top Categories by payment mode
spark.sql("""SELECT
        payment_method,
        COUNT(customer_id) as total_users,
        SUM(total_charges) as mode_revenue,
        AVG(monthly_charges) as avg_monthly_bill
    FROM my_view
    GROUP BY payment_method
    ORDER BY mode_revenue DESC""").show()


+--------------+-----------+-----------------+-----------------+
|payment_method|total_users|     mode_revenue| avg_monthly_bill|
+--------------+-----------+-----------------+-----------------+
|   Credit Card|        279|764829.0800000003|75.92462365591398|
|           UPI|        260|762910.6000000001|82.49053846153845|
| Bank Transfer|        218|654042.5400000004|83.00903669724771|
|          Cash|        243|618453.1600000003|79.17971193415639|
+--------------+-----------+-----------------+-----------------+



In [72]:
# Query 2 : Highest Revenue Segment
spark.sql("""SELECT
        payment_method AS highest_revenue_segment,
        SUM(total_charges) AS total_segment_revenue,
        COUNT(customer_id) AS total_customers
    FROM my_view
    GROUP BY payment_method
    ORDER BY total_segment_revenue DESC
    LIMIT 1""").show()

+-----------------------+---------------------+---------------+
|highest_revenue_segment|total_segment_revenue|total_customers|
+-----------------------+---------------------+---------------+
|            Credit Card|    764829.0800000003|            279|
+-----------------------+---------------------+---------------+



In [74]:
# Query 3: Top 10 Customers with highest Total Charges
spark.sql("""SELECT
customer_id ,total_charges
FROM my_view
ORDER BY total_charges DESC
LIMIT 10""").show()

+-----------+-------------+
|customer_id|total_charges|
+-----------+-------------+
|         95|     10772.52|
|        391|     10558.95|
|        179|     10445.05|
|        519|     10071.46|
|        940|      9940.84|
|        570|      9754.19|
|        939|      9571.84|
|        950|      9465.68|
|        501|      9457.37|
|        481|      9261.92|
+-----------+-------------+



In [80]:
# Query 4 : Monthly Trend Analysis
spark.sql("""
    SELECT
    CASE
        WHEN tenure_months <= 6 THEN '0-6 Months (New)'
        WHEN tenure_months <= 12 THEN '7-12 Months (Infant)'
        WHEN tenure_months <= 24 THEN '13-24 Months (Mid-Tier)'
        ELSE '24+ Months (Loyal)'
    END AS tenure_bucket,
    COUNT(customer_id) AS total_cohort_customers,
    ROUND(AVG(monthly_charges), 2) AS cohort_avg_monthly_charge,
    ROUND(SUM(total_charges), 2) AS cohort_total_revenue
FROM my_view
GROUP BY tenure_bucket
ORDER BY MIN(tenure_months) ASC
""").show()

+--------------------+----------------------+-------------------------+--------------------+
|       tenure_bucket|total_cohort_customers|cohort_avg_monthly_charge|cohort_total_revenue|
+--------------------+----------------------+-------------------------+--------------------+
|    0-6 Months (New)|                    89|                     79.9|            24439.03|
|7-12 Months (Infant)|                    85|                    81.32|            66208.37|
|13-24 Months (Mid...|                   162|                    83.87|           244614.89|
|  24+ Months (Loyal)|                   664|                    78.85|          2464973.09|
+--------------------+----------------------+-------------------------+--------------------+



In [81]:
# Query 5 : Top 10 Records by Business Metric
spark.sql("""
SELECT
    customer_id,
    contract_type,
    internet_service,
    tenure_months,
    monthly_charges,
    ROUND(total_charges, 2) AS lifetime_value
FROM my_view
WHERE churn = 0
ORDER BY total_charges DESC
LIMIT 10
""").show()

+-----------+--------------+----------------+-------------+---------------+--------------+
|customer_id| contract_type|internet_service|tenure_months|monthly_charges|lifetime_value|
+-----------+--------------+----------------+-------------+---------------+--------------+
|        391|      One Year|           Fiber|           69|         149.49|      10558.95|
|        179|Month-to-Month|             DSL|           69|         145.22|      10445.05|
|        950|      One Year|            None|           70|         144.96|       9465.68|
|        501|      One Year|             DSL|           65|         141.16|       9457.37|
|        552|Month-to-Month|             DSL|           67|          132.9|        9145.0|
|        987|      One Year|             DSL|           70|         128.58|       9102.87|
|        723|      One Year|           Fiber|           64|          129.8|       9018.13|
|        804|      One Year|            None|           58|         143.71|       8931.79|

## Task 6: Advanced Transformations
#### Objective :
- Apply enterprise-grade transformations.

In [82]:
silver_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|    true|           58.101|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    true|           83.251|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|    true|            27.64|
|        410| 19|           51|          88.61|     

##### Window Functions - Implement:
- row_number()
- rank()
- dense_rank()
- lag()
- lead()

In [83]:
window_func_df=silver_df.withColumn("row_number",row_number().over(Window.orderBy(col('payment_method'))))\
    .withColumn("rank",rank().over(Window.orderBy(col('payment_method').asc())))\
        .withColumn("dense_rank",dense_rank().over(Window.orderBy(col('payment_method').asc())))\
            .withColumn("lag",lag('payment_method').over(Window.orderBy(col('payment_method').asc())))\
                .withColumn("lead",lead('payment_method').over(Window.orderBy(col('payment_method').asc())))

window_func_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------+----+----------+-------------+-------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|row_number|rank|dense_rank|          lag|         lead|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------+----+----------+-------------+-------------+
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|             DSL|              1| Bank Transfer|    1|    true|          136.339|         1|   1|         1|         NULL|Bank Transfer|
|        129| 65|           25|          15.06|        353.8|Month-to-Month|             DSL|              1| Bank Transfer|    1|    true|         

##### Join Operations - Implement:
- Inner Join
- Left Join

Create a lookup table manually and join with the main dataset.

In [84]:
lookup_data = [("Fiber", "High_Speed"),
               ("DSL", "Medium_Speed"),
                ("None", "No_Internet")]
lookup_df = spark.createDataFrame(lookup_data, ["internet_service", "speed_category"])

#Inner Join
inner_join_df = silver_df.join(lookup_df,silver_df['internet_service'] == lookup_df["internet_service"],'inner')
inner_join_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------+--------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|internet_service|speed_category|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------+--------------+
|        167| 56|            5|          94.88|       517.19|Month-to-Month|           Fiber|              0|           UPI|    0|    true|           86.198|           Fiber|    High_Speed|
|        876| 38|           44|          13.56|       619.29|      One Year|           Fiber|              0|   Credit Card|    0|    true|           13.762|           Fiber|    High_Speed|
|        811| 66|           60|          94.53|   

In [85]:
#Left Join
left_join_df = silver_df.join(lookup_df,silver_df['internet_service'] == lookup_df["internet_service"],'left')
left_join_df.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------+--------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|internet_service|speed_category|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+----------------+--------------+
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    true|           83.251|             DSL|  Medium_Speed|
|        664| 56|           47|         141.36|      6544.26|Month-to-Month|             DSL|              1| Bank Transfer|    1|    true|          136.339|             DSL|  Medium_Speed|
|        861| 51|           14|         138.33|   

## Task 7: Spark ML Pipeline
#### Objective :
- Build a reusable Spark ML Pipeline.

In [86]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

In [89]:
cat_columns = ['contract_type', 'internet_service', 'payment_method']
num_columns = ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets', 'is_adult', 'charge_per_tenure']
ml_df = silver_df.select(cat_columns + num_columns + ["churn"]).dropna()

##### StringIndexer :
- StringIndexer()

In [91]:
# String Indexer
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in cat_columns]
indexers

[StringIndexer_b58e772a6469,
 StringIndexer_9b72d29cc585,
 StringIndexer_a5805757a814]

##### OneHotEncoder :
- OneHotEncoder()

In [93]:
#OneHotEncoder
encoders = [OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_vec") for c in cat_columns]
encoders

[OneHotEncoder_262ccae67d80,
 OneHotEncoder_eb4d7e30758c,
 OneHotEncoder_acb3536d7de1]

##### VectorAssembler :
- VectorAssembler()

In [94]:
#VectorAssembler
assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in cat_columns] + num_columns,
    outputCol="features"
)
assembler

VectorAssembler_eed4ed0c14c6

##### Model (Choose one):
- Linear Regression
- Logistic Regression
- Random Forest Classifier
- Decision Tree Classifier

In [95]:
#Logistic Regression
log_reg = LogisticRegression(featuresCol="features", labelCol="churn",predictionCol="prediction")
log_reg

LogisticRegression_84961a7b7dae

##### Pipeline :
- Pipeline()

In [96]:
# Pipeline
pipeline = Pipeline(stages = indexers + encoders + [assembler, log_reg])
pipeline

Pipeline_2f09dc7c7c9f

In [97]:
# Train Test Split
train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)
train_data.head(5)

[Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=18, tenure_months=3, monthly_charges=136.9, total_charges=393.86, support_tickets=4, is_adult=True, charge_per_tenure=98.465, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=18, tenure_months=7, monthly_charges=94.08, total_charges=689.0, support_tickets=3, is_adult=True, charge_per_tenure=86.125, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=21, tenure_months=68, monthly_charges=57.94, total_charges=4039.82, support_tickets=3, is_adult=True, charge_per_tenure=58.548, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=23, tenure_months=4, monthly_charges=86.02, total_charges=350.57, support_tickets=1, is_adult=True, charge_per_tenure=70.114, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', paymen

In [98]:
test_data.head(5)

[Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=21, tenure_months=56, monthly_charges=28.55, total_charges=1477.38, support_tickets=3, is_adult=True, charge_per_tenure=25.919, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=19, monthly_charges=142.17, total_charges=2641.41, support_tickets=1, is_adult=True, charge_per_tenure=132.07, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=66, monthly_charges=40.34, total_charges=2584.42, support_tickets=2, is_adult=True, charge_per_tenure=38.573, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=31, tenure_months=9, monthly_charges=111.99, total_charges=967.33, support_tickets=0, is_adult=True, charge_per_tenure=96.733, churn=1),
 Row(contract_type='Month-to-Month', internet_service='DSL',

In [99]:
# Fit Model
print("Training Model")
model = pipeline.fit(train_data)
predictions = model.transform(test_data)
predictions.head(5)

Training Model


[Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=21, tenure_months=56, monthly_charges=28.55, total_charges=1477.38, support_tickets=3, is_adult=True, charge_per_tenure=25.919, churn=1, contract_type_index=0.0, internet_service_index=1.0, payment_method_index=3.0, contract_type_vec=SparseVector(3, {0: 1.0}), internet_service_vec=SparseVector(3, {1: 1.0}), payment_method_vec=SparseVector(4, {3: 1.0}), features=SparseVector(17, {0: 1.0, 4: 1.0, 9: 1.0, 10: 21.0, 11: 56.0, 12: 28.55, 13: 1477.38, 14: 3.0, 15: 1.0, 16: 25.919}), rawPrediction=DenseVector([-0.0966, 0.0966]), probability=DenseVector([0.4759, 0.5241]), prediction=1.0),
 Row(contract_type='Month-to-Month', internet_service='DSL', payment_method='Bank Transfer', age=26, tenure_months=19, monthly_charges=142.17, total_charges=2641.41, support_tickets=1, is_adult=True, charge_per_tenure=132.07, churn=1, contract_type_index=0.0, internet_service_index=1.0, payment_method_index=3.0, c

In [100]:
# quick preview of actual vs predicted churn values
predictions.select("churn", "prediction", "probability").show(5)

+-----+----------+--------------------+
|churn|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.47586550453544...|
|    1|       1.0|[0.18365068991906...|
|    1|       0.0|[0.55881969439270...|
|    1|       1.0|[0.25805909763853...|
|    1|       1.0|[0.22584907063585...|
+-----+----------+--------------------+
only showing top 5 rows


## Task 8: Performance Optimization
#### Objective :
- Optimize Spark Jobs.

In [108]:
import time
start_time = time.time()
unoptimized_agg_time = time.time() - start_time

In [109]:
# Unoptimized Operation (Directly from File)
start_time = time.time()
spark.read.parquet("silver").groupBy("contract_type").agg(avg("monthly_charges")).collect()
unoptimized_agg_time = time.time() - start_time
unoptimized_agg_time

0.2650160789489746

In [110]:
#Cache
df_opt = spark.read.parquet("silver")
df_opt.cache()
df_opt.count()
start_time = time.time()
optimized_agg_time = time.time() - start_time
optimized_agg_time

7.82012939453125e-05

In [111]:
# Optimization 2: Repartitioning
df_repartitioned = df_opt.repartition(8)
df_repartitioned.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|is_adult|charge_per_tenure|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+--------+-----------------+
|        650| 64|           32|          19.75|        677.8|Month-to-Month|            None|              1| Bank Transfer|    0|    true|           20.539|
|        485| 43|           10|          72.96|       696.09|      Two Year|           Fiber|              3| Bank Transfer|    0|    true|           63.281|
|        344| 20|           26|         141.23|      3816.08|Month-to-Month|           Fiber|              1|          Cash|    0|    true|          141.336|
|        621| 29|           71|         109.43|     

In [112]:
#Optimization 3: Broadcast Join
start_time = time.time()
df_opt.join(broadcast(lookup_df), "internet_service").count()#type:ignore
broadcast_join_time = time.time() - start_time
broadcast_join_time

0.8396940231323242

In [113]:
print(f"Unoptimized Aggregation Time: {unoptimized_agg_time:.4f} seconds")
print(f"Optimized (Cached) Aggregation Time: {optimized_agg_time:.4f} seconds")
print(f"Broadcast Join Time: {broadcast_join_time:.4f} seconds")

Unoptimized Aggregation Time: 0.2650 seconds
Optimized (Cached) Aggregation Time: 0.0001 seconds
Broadcast Join Time: 0.8397 seconds
